# **Utility Fuctions**

In [23]:
import numpy as np
import cv2 as cv


def countFrames(path):
    video=cv.VideoCapture(path)
    count=0
    while True:
        ret,frame=video.read()
        if not ret: break
        count=count+1

    video.release()
    return count


def sneakPeekVideo(frames):
    for i in range(16):
        cv.imshow('frame', frames[i])
        cv.waitKey(500)  
    cv.destroyAllWindows()
    


def loadClip(path):
    video = cv.VideoCapture(path)
    count = countFrames(path)
    idxs = np.linspace(0, count-1, 16).astype(int)

    frames = []
    for i in idxs:
        video.set(cv.CAP_PROP_POS_FRAMES, i)
        ret, frame = video.read()
        if ret:
            frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            frames.append(frame)

    video.release()
    return frames   
    

In [24]:
import random

def augment(frames):
    # frames: numpy (16, 112, 112, 3) float32, RGB, 0-1

    # 1. horizontal flip (50% chance)
    if random.random() < 0.5:
        frames = frames[:, :, ::-1, :].copy()

    # 2. random brightness / contrast jitter
    if random.random() < 0.5:
        gain = random.uniform(0.7, 1.3)
        frames = np.clip(frames * gain, 0, 1)

    # 3. random crop: take a random 96x96 window, resize back to 112x112
    if random.random() < 0.5:
        h, w = 96, 96
        top = random.randint(0, 112 - h)
        left = random.randint(0, 112 - w)
        cropped = frames[:, top:top+h, left:left+w, :]
        frames = np.array([cv.resize(f, (112, 112)) for f in cropped])

    return frames

# **Datasets**

In [25]:
import torch, glob
import os
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

checkpoint = "MCG-NJU/videomae-base-finetuned-kinetics"
processor = VideoMAEImageProcessor.from_pretrained(checkpoint)

class ActionDataset(torch.utils.data.Dataset):
    def __init__(self, root):
        self.classes = {"wave":0, "walk":1, "sitting":2, "standing":3}
        self.items = []

        paths = glob.glob(root + "/*/*.mp4")
        for path in paths:
            names = path.split(os.sep)
            label = self.classes[names[-2]]
            
            # Processor resizes to 224x224, normalizes, and outputs (1, 16, 3, 224, 224)
            inputs = processor(loadClip(path), return_tensors="pt")
            video = inputs["pixel_values"].squeeze(0)  # becomes (16, 3, 224, 224)
            
            self.items.append((video, label))

    def __len__(self):
        return len(self.items)
        
    def __getitem__(self, i):
        return self.items[i]

In [26]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| GPU count:", torch.cuda.device_count())

Using device: cuda | GPU count: 2


In [27]:
trainDs = ActionDataset("/kaggle/input/datasets/zadifmustafa/trainingdata/data")

loader = torch.utils.data.DataLoader(trainDs, batch_size=4, shuffle=True)
testDs = ActionDataset("/kaggle/input/datasets/zadifmustafa/testingdata/test")
testLoader = torch.utils.data.DataLoader(testDs, batch_size=4, shuffle=False)



In [28]:
print(len(trainDs))    # expect 62
print(len(testDs))     # expect 20

66
16


# **Testing**

In [29]:
import numpy as np
from sklearn.metrics import confusion_matrix
import torch

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for clips, labels in loader:
            clips = clips.to(device)

            out = model(clips).logits
            preds = torch.argmax(out, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    matrix = confusion_matrix(all_labels, all_preds)
    accuracy = matrix.trace() / matrix.sum()
    return accuracy, matrix

# **Training**

In [33]:
import torch, glob
import os
import torch.nn as nn
# Replace `model = CNN().to(device)` with:
model = VideoMAEForVideoClassification.from_pretrained(
    checkpoint,
    num_labels=4,
    ignore_mismatched_sizes=True
).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(15):
    model.train()
    for clips, labels in loader:
        clips = clips.to(device)
        labels = labels.to(device)

        out = model(clips).logits
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc, matrix = evaluate(model, testLoader)
    print(f"epoch {epoch}, loss {loss.item():.4f}, test accuracy {acc:.3f}")

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `400`.


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key               | Status   |                                                                                         
------------------+----------+-----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([400, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


epoch 0, loss 0.0854, test accuracy 1.000
epoch 1, loss 0.0123, test accuracy 1.000
epoch 2, loss 0.0062, test accuracy 1.000
epoch 3, loss 0.0042, test accuracy 1.000
epoch 4, loss 0.0029, test accuracy 1.000
epoch 5, loss 0.0024, test accuracy 1.000
epoch 6, loss 0.0021, test accuracy 1.000
epoch 7, loss 0.0017, test accuracy 1.000
epoch 8, loss 0.0015, test accuracy 1.000
epoch 9, loss 0.0010, test accuracy 1.000
epoch 10, loss 0.0011, test accuracy 1.000
epoch 11, loss 0.0009, test accuracy 1.000
epoch 12, loss 0.0008, test accuracy 1.000
epoch 13, loss 0.0008, test accuracy 1.000
epoch 14, loss 0.0007, test accuracy 1.000


In [34]:
train_acc, train_matrix = evaluate(model, loader)
test_acc,  test_matrix  = evaluate(model, testLoader)
print(f"train: {train_acc:.3f}   test: {test_acc:.3f}")

# the leak test
train_paths = set(p for p, _ in trainDs.items)
test_paths  = set(p for p, _ in testDs.items)
print("overlap:", len(train_paths & test_paths))

# and the class balance of each set
from collections import Counter
print("train classes:", Counter(l for _, l in trainDs.items))
print("test  classes:", Counter(l for _, l in testDs.items))

train: 1.000   test: 1.000
overlap: 0
train classes: Counter({2: 17, 3: 17, 1: 17, 0: 15})
test  classes: Counter({2: 4, 3: 4, 1: 4, 0: 4})


In [ ]:
torch.save(model.state_dict(), "cnn_model.pth")

In [35]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get predictions on the test dataset
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for clips, labels in testLoader:
        clips = clips.to(device)
        outputs = model(clips)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Your notebook's class-to-label mapping
class_names = ["wave", "walk", "sitting", "standing"]

# Calculate confusion matrix
cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3])

# Plot it
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("3D CNN - Test Confusion Matrix")
plt.tight_layout()
plt.show()

TypeError: argmax(): argument 'input' (position 1) must be Tensor, not ImageClassifierOutput

In [36]:
train_acc, _ = evaluate(model, loader)
print("train acc:", train_acc)
print("params:", sum(p.numel() for p in model.parameters()))
torch.save(model.state_dict(), "cnn_model.pth")

train acc: 1.0
params: 86230276
